In [12]:
# Library yang digunakan
import pandas as pd
import re

## 💡 Load Data

In [13]:
#Load Dataser
file_path = "../data/review_genshin_5.5.csv"
df = pd.read_csv(file_path)
version = '5.5'

In [14]:
# Show Dataset
df.head(10)

,ReviewID,Username,UserImage,Content,Score,ThumbsUpCount,ReviewCreatedVersion,at,ReplyContent,RepliedAt,AppVersion
0,2cf85c2d-bc47-4030-9eca-ac000d46d82f,Dwi Surya,https://play-lh.googleusercontent.com/a-/ALV-U...,HELLO GENSHIN IMPACT I HAVE A LITTLE COMPLAINT...,5,0,NaN,2025-04-10 18:30:38,NaN,NaN,NaN
1,c97ec484-677d-4d17-b330-39502dd9b045,Roni Saleh Ardiansyah,https://play-lh.googleusercontent.com/a-/ALV-U...,"Saran buat dev dioptimalkan lagi game ya, 2020...",1,0,NaN,2025-04-09 20:08:42,NaN,NaN,NaN
2,8109b41a-4714-4390-b00c-c3cbe005462f,Moriansyah Dinamoria Putra,https://play-lh.googleusercontent.com/a-/ALV-U...,Keren,1,0,2.6.0_6179196_6305792,2025-04-09 20:06:01,NaN,NaN,2.6.0_6179196_6305792
3,4550d4ff-d199-4983-81f5-a299b923ab19,Roger Grahan,https://play-lh.googleusercontent.com/a/ACg8oc...,good,5,0,NaN,2025-04-09 19:57:42,NaN,NaN,NaN
4,4318f7e7-e90b-4fd8-acae-45a2dd6e7452,Diana Majid,https://play-lh.googleusercontent.com/a-/ALV-U...,game nya asik bet anjay,5,0,NaN,2025-04-09 19:55:05,NaN,NaN,NaN
5,219753dc-bfa0-46c5-84d8-7e6c69bfc13e,Muhamad Hilal,https://play-lh.googleusercontent.com/a-/ALV-U...,bisa tidak sih kasih mata buat F2P? mereka ter...,1,10,5.3.0_29183395_29332470,2025-04-09 19:53:58,"Halo, Traveler! Traveler dapat melihat aturan ...",2025-02-18 18:05:21,5.3.0_29183395_29332470
6,fa38b679-0c26-4922-82c5-08474ed35fed,Nahda Dd,https://play-lh.googleusercontent.com/a/ACg8oc...,Ini game atau seni yaallah soalnya bgus bgt!!!...,5,0,NaN,2025-04-09 19:46:49,NaN,NaN,NaN
7,0c916f46-3643-41fe-a684-fa4f9a2e3dad,zaky,https://play-lh.googleusercontent.com/a/ACg8oc...,good game,5,0,NaN,2025-04-09 19:32:24,NaN,NaN,NaN
8,65b278f4-fb78-477b-8368-dd5f310f6d27,Ahmad Robbi,https://play-lh.googleusercontent.com/a-/ALV-U...,oke saya tidak ada lagi yang ingin di sampaika...,5,211,5.5.0_31400259_31451966,2025-04-09 19:26:42,NaN,NaN,5.5.0_31400259_31451966
9,5708cf2e-fe86-4db2-bba1-2da54f464898,Queensha Aqilasunnyhanin,https://play-lh.googleusercontent.com/a/ACg8oc...,game ny oke tp g tau np itu skrg klo mau downl...,3,0,5.5.0_31400259_31451966,2025-04-09 19:10:07,NaN,NaN,5.5.0_31400259_31451966


| Kolom                | Deskripsi                                 |
|----------------------|--------------------------------------------|
| `ReviewId`           | ID unik untuk review                       |
| `UserName`           | Nama pengguna                              |
| `UserImage`          | URL foto profil pengguna                   |
| `Content`            | Isi review                                 |
| `Score`              | Nilai rating (1–5)                         |
| `ThumbsUpCount`      | Jumlah likes pada review                   |
| `ReviewCreatedVersion` | Versi aplikasi saat review dibuat        |
| `at`                 | Tanggal review                             |
| `ReplyContent`       | Balasan dari developer (jika ada)          |
| `RepliedAt`          | Tanggal balasan                            |
| `AppVersion`         | Versi aplikasi                             |

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1282 entries, 0 to 1281
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ReviewID              1282 non-null   object
 1   Username              1282 non-null   object
 2   UserImage             1282 non-null   object
 3   Content               1282 non-null   object
 4   Score                 1282 non-null   int64 
 5   ThumbsUpCount         1282 non-null   int64 
 6   ReviewCreatedVersion  887 non-null    object
 7   at                    1282 non-null   object
 8   ReplyContent          28 non-null     object
 9   RepliedAt             28 non-null     object
 10  AppVersion            887 non-null    object
dtypes: int64(2), object(9)
memory usage: 110.3+ KB


## 🗃️ Drop Kolom

In [16]:
df_cleaned = df.drop(
    columns=['Username', 'UserImage', 'ReplyContent', 'RepliedAt', 'ReviewCreatedVersion'])

Kolom 'userName' dan 'userImage' di-drop karena tidak mengandung informasi tekstual yang relevan untuk analisis NLP — mereka hanya metadata pengguna.  
Kolom 'replyContent' dan 'repliedAt' juga tidak digunakan karena fokus analisis NLP berada pada konten utama, bukan pada balasan atau timestamp-nya.
Kolom 'ReviewCreatedVersion' juga di-drop karena informasinya redundan — nilainya sama dengan kolom 'appVersion'.

## 🔄 Ubah Tipe Data

In [17]:
df_cleaned['at'] = pd.to_datetime(df_cleaned['at'], errors='coerce')

## 🔄 Transform Nilai Data

In [18]:
def version_simplyfied(v):
    if pd.isna(v):
        return version
    match = re.match(r"^([0-9]+)\.([0-9]+)", str(v))
    if match:
        return f"{match.group(1)}.{match.group(2)}"
    return version  # fallback jika tidak cocok juga


df_cleaned['AppVersion'] = df_cleaned['AppVersion'].apply(version_simplyfied)

## 🔍 Cek Outlier / Nilai Tidak Normal

In [19]:
df_filtered = df_cleaned[df_cleaned['AppVersion'] == version]

In [20]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 964 entries, 0 to 1277
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ReviewID       964 non-null    object        
 1   Content        964 non-null    object        
 2   Score          964 non-null    int64         
 3   ThumbsUpCount  964 non-null    int64         
 4   at             964 non-null    datetime64[ns]
 5   AppVersion     964 non-null    object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 52.7+ KB


In [21]:
df_filtered.head()

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
0,2cf85c2d-bc47-4030-9eca-ac000d46d82f,HELLO GENSHIN IMPACT I HAVE A LITTLE COMPLAINT...,5,0,2025-04-10 18:30:38,5.5
1,c97ec484-677d-4d17-b330-39502dd9b045,"Saran buat dev dioptimalkan lagi game ya, 2020...",1,0,2025-04-09 20:08:42,5.5
3,4550d4ff-d199-4983-81f5-a299b923ab19,good,5,0,2025-04-09 19:57:42,5.5
4,4318f7e7-e90b-4fd8-acae-45a2dd6e7452,game nya asik bet anjay,5,0,2025-04-09 19:55:05,5.5
6,fa38b679-0c26-4922-82c5-08474ed35fed,Ini game atau seni yaallah soalnya bgus bgt!!!...,5,0,2025-04-09 19:46:49,5.5


## 🗃️ Save Cleaned Data

In [22]:
# Simpan ke CSV
output_path = f'../data/review_genshin_{version}_cleaned.csv'
df_filtered.to_csv(output_path, index=False, encoding='utf-8-sig')